In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
 
warnings.filterwarnings("ignore")

In [2]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)


In [3]:
LOYALTY_FILE  = "C:\Desktop\Airline\Dataset\Customer Loyalty History.csv"
FLIGHTS_FILE  = "C:\Desktop\Airline\Dataset\Customer Flight Activity.csv"
CALENDAR_FILE = "C:\Desktop\Airline\Dataset\Calendar.csv"
 
OUTPUT_DIR    = "Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
# We will log every cleaning decision here
cleaning_log = []
 
def log(issue, decision, reason):
    """Record a cleaning decision for the technical report."""
    cleaning_log.append({"Issue": issue, "Decision": decision, "Reason": reason})
    print(f"  [DECISION] {issue} → {decision}")
 

In [5]:
print("\n" + "="*70)
print("STEP 1 — LOADING DATA")
print("="*70)
 
loyalty  = pd.read_csv(LOYALTY_FILE)
flights  = pd.read_csv(FLIGHTS_FILE)
calendar = pd.read_csv(CALENDAR_FILE)
 
print(f"  Loyalty  : {loyalty.shape[0]:,} rows × {loyalty.shape[1]} cols")
print(f"  Flights  : {flights.shape[0]:,} rows × {flights.shape[1]} cols")
print(f"  Calendar : {calendar.shape[0]:,} rows × {calendar.shape[1]} cols")
 


STEP 1 — LOADING DATA
  Loyalty  : 16,737 rows × 16 cols
  Flights  : 392,936 rows × 8 cols
  Calendar : 2,557 rows × 4 cols


In [6]:
print("\n" + "="*70)
print("STEP 2 — COLUMN NAMES & DTYPES")
print("="*70)
 
print("\n── Loyalty History ──")
print(loyalty.dtypes.to_string())
 
print("\n── Flight Activity ──")
print(flights.dtypes.to_string())
 
print("\n── Calendar ──")
print(calendar.dtypes.to_string())


STEP 2 — COLUMN NAMES & DTYPES

── Loyalty History ──
Loyalty Number          int64
Country                object
Province               object
City                   object
Postal Code            object
Gender                 object
Education              object
Salary                float64
Marital Status         object
Loyalty Card           object
CLV                   float64
Enrollment Type        object
Enrollment Year         int64
Enrollment Month        int64
Cancellation Year     float64
Cancellation Month    float64

── Flight Activity ──
Loyalty Number                   int64
Year                             int64
Month                            int64
Total Flights                    int64
Distance                         int64
Points Accumulated             float64
Points Redeemed                  int64
Dollar Cost Points Redeemed      int64

── Calendar ──
Date                object
Start of Year       object
Start of Quarter    object
Start of Month      object


In [7]:
def clean_cols(df):
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace(" ", "_")
                  .str.replace(r"[^a-z0-9_]", "", regex=True))
    return df
 
loyalty  = clean_cols(loyalty)
flights  = clean_cols(flights)
calendar = clean_cols(calendar)
 
print("\n  Column names normalised (lowercase + underscores).")
 


  Column names normalised (lowercase + underscores).


In [8]:
loyalty.head()

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,NaN,Single,Star,3839.75,Standard,2014,7,2018.0,1.0
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,NaN,Single,Star,3839.75,Standard,2013,2,NaN,NaN
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,NaN,NaN


In [9]:
print(f"\n  Raw loyalty shape : {loyalty.shape}")
print(f"\n  enrollment_year sample (raw):")
print(loyalty["enrollment_year"].value_counts().sort_index().to_string())
 


  Raw loyalty shape : (16737, 16)

  enrollment_year sample (raw):
enrollment_year
2012    1686
2013    2397
2014    2370
2015    2331
2016    2456
2017    2487
2018    3010


In [10]:
print(f"  enrollment_year dtype  : {loyalty['enrollment_year'].dtype} :  {loyalty['enrollment_year'][0]}")
print(f"  enrollment_month dtype : {loyalty['enrollment_month'].dtype} : {loyalty['enrollment_month'][0]}")
print(f"  cancellation_year dtype: {loyalty['cancellation_year'].dtype} : {loyalty['cancellation_year'][0]}")
 

  enrollment_year dtype  : int64 :  2016
  enrollment_month dtype : int64 : 2
  cancellation_year dtype: float64 : nan


In [11]:
loyalty.isnull().sum()

loyalty_number            0
country                   0
province                  0
city                      0
postal_code               0
gender                    0
education                 0
salary                 4238
marital_status            0
loyalty_card              0
clv                       0
enrollment_type           0
enrollment_year           0
enrollment_month          0
cancellation_year     14670
cancellation_month    14670
dtype: int64

In [12]:
print("\n" + "="*70)
print("STEP 3 — MISSING VALUES")
print("="*70)
 
def missing_report(df, name):
    miss = df.isnull().sum()
    miss = miss[miss > 0]
    if miss.empty:
        print(f"\n  {name}: No missing values")
    else:
        pct = (miss / len(df) * 100).round(2)
        report = pd.DataFrame({"missing_count": miss, "missing_pct": pct})
        print(f"\n  {name} — missing values:")
        print(report.to_string())
    return miss
 
loyalty_miss  = missing_report(loyalty,  "Loyalty History")
flights_miss  = missing_report(flights,  "Flight Activity")
 


STEP 3 — MISSING VALUES

  Loyalty History — missing values:
                    missing_count  missing_pct
salary                       4238        25.32
cancellation_year           14670        87.65
cancellation_month          14670        87.65

  Flight Activity: No missing values


In [13]:
loyalty["cancellation_year"].notna().sum()

np.int64(2067)

In [14]:
# ── Handle missing values in Loyalty ─────────────────────────────────────────
 
# Salary band
if "salary" in loyalty.columns or any("salary" in c for c in loyalty.columns):
    sal_col = [c for c in loyalty.columns if "salary" in c][0]
    n_miss = loyalty[sal_col].isnull().sum()
    if n_miss > 0:
        loyalty[sal_col] = loyalty[sal_col].fillna("Unknown")
        log(f"Missing {sal_col} ({n_miss} rows)",
            "Filled with 'Unknown' category",
            "Salary band is categorical; imputing median would imply false precision")
 

# Cancellation date — expected to be null for active members; leave as-is
if any("cancel" in c for c in loyalty.columns):
    cancel_col = [c for c in loyalty.columns if "cancel" in c][0]
    n_cancel = loyalty[cancel_col].notna().sum()
    print(f"\n  Members with formal cancellation: {n_cancel:,} "
          f"({n_cancel/len(loyalty)*100:.1f}%)")
    log(f"Null {cancel_col}",
        "Kept as null — indicates active member",
        "Null cancellation date means member has NOT cancelled; this is valid data")

  [DECISION] Missing salary (4238 rows) → Filled with 'Unknown' category

  Members with formal cancellation: 2,067 (12.3%)
  [DECISION] Null cancellation_year → Kept as null — indicates active member


In [15]:
print("\n" + "="*70)
print("STEP 4 — DATE PARSING")
print("="*70)
 
# Build proper date columns from integer year/month
loyalty["enrollment_date"] = pd.to_datetime(
    dict(year  = loyalty["enrollment_year"],
         month = loyalty["enrollment_month"],
         day   = 1),
    errors="coerce"
)
print(f"  Built enrollment_date: {loyalty['enrollment_date'].min().date()} → "
      f"{loyalty['enrollment_date'].max().date()}")
 
has_cancel = loyalty["cancellation_year"].notna()
loyalty["cancellation_date"] = pd.NaT
loyalty.loc[has_cancel, "cancellation_date"] = pd.to_datetime(
    dict(year  = loyalty.loc[has_cancel, "cancellation_year"],
         month = loyalty.loc[has_cancel, "cancellation_month"].fillna(12),
         day   = 1),
    errors="coerce"
)
print(f"  Built cancellation_date — "
      f"{has_cancel.sum():,} members have a cancellation date")
 
# Flights: year + month columns → create a period/date column
year_col  = [c for c in flights.columns if "year"  in c]
month_col = [c for c in flights.columns if "month" in c]
 
if year_col and month_col:
    year_col, month_col = year_col[0], month_col[0]
    flights["activity_date"] = pd.to_datetime(
        dict(year=flights[year_col], month=flights[month_col], day=1),
        errors="coerce"
    )
    print(f"  Created 'activity_date' from {year_col} + {month_col}")
    print(f"  Date range: {flights['activity_date'].min()} → "
          f"{flights['activity_date'].max()}")
    log("No single date column in flights",
        "Combined year + month into activity_date (YYYY-MM-01)",
        "Required for time-series feature engineering in later phases")
 
 


STEP 4 — DATE PARSING
  Built enrollment_date: 2012-04-01 → 2018-12-01
  Built cancellation_date — 2,067 members have a cancellation date
  Created 'activity_date' from year + month
  Date range: 2017-01-01 00:00:00 → 2018-12-01 00:00:00
  [DECISION] No single date column in flights → Combined year + month into activity_date (YYYY-MM-01)


In [16]:
loyalty.head()

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month,enrollment_date,cancellation_date
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN,2016-02-01,NaT
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,Unknown,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN,2016-03-01,NaT
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,Unknown,Single,Star,3839.75,Standard,2014,7,2018.0,1.0,2014-07-01,2018-01-01
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,Unknown,Single,Star,3839.75,Standard,2013,2,NaN,NaN,2013-02-01,NaT
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,NaN,NaN,2014-10-01,NaT


In [62]:
flights.head()

,loyalty_number,year,month,total_flights,distance,points_accumulated,points_redeemed,dollar_cost_points_redeemed,activity_date
0,100590,2018,6,12,15276,22914.0,0,0,2018-06-01
1,100590,2018,7,12,9168,13752.0,0,0,2018-07-01
2,100590,2018,5,4,6504,9756.0,0,0,2018-05-01
3,100590,2018,10,0,0,0.0,512,92,2018-10-01
4,100590,2018,2,0,0,0.0,0,0,2018-02-01


In [17]:
print("\n" + "="*70)
print("STEP 5 — DUPLICATE ROWS")
print("="*70)
 
for df, name in [(loyalty, "Loyalty"), (flights, "Flights")]:
    n_dup = df.duplicated().sum()
    print(f"  {name}: {n_dup} exact duplicate rows")
    if n_dup > 0:
        df.drop_duplicates(inplace=True)
        log(f"{n_dup} duplicate rows in {name}",
            "Dropped exact duplicates",
            "Exact duplicates are data entry errors; retaining would inflate counts")
 
# Member-month duplicates in flights (same member, same year-month)
id_col = "loyalty_number"
 
dup_member_month = flights.duplicated(
    subset=[id_col, year_col, month_col], keep=False
).sum()
print(f"\n  Flights: {dup_member_month} rows with duplicate member-month "
      f"(same member, same year-month)")
if dup_member_month > 0:
    flights = (flights
               .groupby([id_col, year_col, month_col, "activity_date"])
               .sum(numeric_only=True)
               .reset_index())
    log(f"{dup_member_month} duplicate member-month rows in flights",
        "Aggregated by summing numeric columns",
        "A member cannot have two separate records for the same month; "
        "sum is the correct aggregation for counts and distances")
 


STEP 5 — DUPLICATE ROWS
  Loyalty: 0 exact duplicate rows
  Flights: 1922 exact duplicate rows
  [DECISION] 1922 duplicate rows in Flights → Dropped exact duplicates

  Flights: 3892 rows with duplicate member-month (same member, same year-month)
  [DECISION] 3892 duplicate member-month rows in flights → Aggregated by summing numeric columns


In [18]:
print("\n" + "="*70)
print("STEP 6 — NUMERIC ANOMALIES")
print("="*70)
 
# ── Flight Activity numeric columns ──────────────────────────────────────────
numeric_flight_cols = flights.select_dtypes(include=[np.number]).columns.tolist()
# Remove year/month from anomaly check
numeric_flight_cols = [c for c in numeric_flight_cols
                       if c not in [year_col, month_col]]
 
print("\n  Flight Activity — descriptive stats:")
print(flights[numeric_flight_cols].describe().round(2).to_string())
 
# Negative values check
for col in numeric_flight_cols:
    n_neg = (flights[col] < 0).sum()
    if n_neg > 0:
        print(f"\n  WARNING: {n_neg} negative values in '{col}'")
        flights[col] = flights[col].clip(lower=0)
        log(f"Negative values in flights['{col}'] ({n_neg} rows)",
            "Clipped to 0 (lower bound)",
            "Negative flights/distances/points are physically impossible; "
            "likely data entry errors")
 
# Points redeemed > points earned (at member level)
pts_earned_col  = "points_accumulated"
pts_redeem_col  = "points_redeemed"
 
if pts_earned_col and pts_redeem_col:
    member_totals = (flights.groupby(id_col)[[pts_earned_col, pts_redeem_col]].sum())
    n_over_redeem = (member_totals[pts_redeem_col] > member_totals[pts_earned_col]).sum()
    print(f"\n  Members redeeming more points than earned: {n_over_redeem}")
    if n_over_redeem > 0:
        log(f"{n_over_redeem} members redeem more points than earned (lifetime)",
            "Flagged with column 'points_anomaly_flag'; kept in dataset",
            "Could indicate points transferred from partner programs; "
            "removing would lose valid members")
        anomaly_ids = member_totals[
            member_totals[pts_redeem_col] > member_totals[pts_earned_col]
        ].index
        flights["points_anomaly_flag"] = flights[id_col].isin(anomaly_ids).astype(int)
 
# ── CLV check ────────────────────────────────────────────────────────────────
clv_col = [c for c in loyalty.columns if "clv" in c or "lifetime" in c]
if clv_col:
    clv_col = clv_col[0]
    n_neg_clv = (loyalty[clv_col] < 0).sum()
    n_zero_clv = (loyalty[clv_col] == 0).sum()
    print(f"\n  CLV — negative: {n_neg_clv}, zero: {n_zero_clv}")
    print(f"  CLV — mean: {loyalty[clv_col].mean():.2f}, "
          f"median: {loyalty[clv_col].median():.2f}, "
          f"max: {loyalty[clv_col].max():.2f}")
    if n_neg_clv > 0:
        log(f"{n_neg_clv} negative CLV values",
            "Kept as-is; flagged with 'negative_clv_flag'",
            "Negative CLV is possible for members who redeemed more than earned; "
            "dropping them would bias the segmentation")
        loyalty["negative_clv_flag"] = (loyalty[clv_col] < 0).astype(int)
 


STEP 6 — NUMERIC ANOMALIES

  Flight Activity — descriptive stats:
       loyalty_number  total_flights   distance  points_accumulated  points_redeemed  dollar_cost_points_redeemed
count       389065.00      389065.00  389065.00           389065.00        389065.00                    389065.00
mean        550237.48           1.31    1960.76             2047.34            31.62                         5.69
std         258565.82           1.98    3262.19             3897.26           127.29                        22.92
min         100018.00           0.00       0.00                0.00             0.00                         0.00
25%         327470.00           0.00       0.00                0.00             0.00                         0.00
50%         551510.00           0.00       0.00                0.00             0.00                         0.00
75%         772019.00           2.00    3056.00             3078.00             0.00                         0.00
max         999986.0

In [19]:
flights.head()

,loyalty_number,year,month,activity_date,total_flights,distance,points_accumulated,points_redeemed,dollar_cost_points_redeemed,points_anomaly_flag
0,100018,2017,1,2017-01-01,1,601,601.0,0,0,0
1,100018,2017,2,2017-02-01,0,0,0.0,0,0,0
2,100018,2017,3,2017-03-01,4,9648,9648.0,438,79,0
3,100018,2017,4,2017-04-01,1,1654,1654.0,0,0,0
4,100018,2017,5,2017-05-01,0,0,0.0,0,0,0


In [20]:
flights["points_anomaly_flag"].value_counts()

points_anomaly_flag
0    388801
1       264
Name: count, dtype: int64

In [21]:
print("\n" + "="*70)
print("STEP 7 — MEMBER COVERAGE ACROSS TABLES")
print("="*70)
 
loyalty_id_col = "loyalty_number"
 
loyalty_ids = set(loyalty[loyalty_id_col])
flights_ids = set(flights[id_col])
 
only_loyalty = loyalty_ids - flights_ids
only_flights = flights_ids - loyalty_ids
both         = loyalty_ids & flights_ids
 
print(f"  Members in Loyalty only (no flight records) : {len(only_loyalty):,}")
print(f"  Members in Flights only (no loyalty record) : {len(only_flights):,}")
print(f"  Members in both tables                      : {len(both):,}")
 
if len(only_loyalty) > 0:
    log(f"{len(only_loyalty)} members in Loyalty with no flight records",
        "Kept in loyalty table; excluded from flight-based features (will be NaN → 0)",
        "These are likely enrolled-but-never-flew members; "
        "important for churn analysis as extreme inactives")
 
if len(only_flights) > 0:
    log(f"{len(only_flights)} members in Flights with no loyalty record",
        "Dropped from flights table",
        "Cannot analyse members without demographic/CLV data; "
        "likely orphaned records from a system migration")
    flights = flights[flights[id_col].isin(loyalty_ids)]
 


STEP 7 — MEMBER COVERAGE ACROSS TABLES
  Members in Loyalty only (no flight records) : 0
  Members in Flights only (no loyalty record) : 0
  Members in both tables                      : 16,737


In [22]:
print("\n" + "="*70)
print("STEP 8 — ACTIVITY PATTERNS")
print("="*70)
 
# Members with zero total flights
flights_col = ["total_flights"]

if flights_col:
    flights_col = flights_col[0]
    member_flight_totals = flights.groupby(id_col)[flights_col].sum()
    n_zero_fliers = (member_flight_totals == 0).sum()
    print(f"  Members with zero total flights: {n_zero_fliers:,}")
    log(f"{n_zero_fliers} members with zero total flights over full period",
        "Kept; tagged as 'never_flew' flag in loyalty table",
        "Never-flew members are a distinct churn-risk segment; "
        "removing them would undercount at-risk population")
    zero_ids = member_flight_totals[member_flight_totals == 0].index
    loyalty["never_flew_flag"] = loyalty[loyalty_id_col].isin(zero_ids).astype(int)
    print(f"  'never_flew_flag' added to loyalty table.")
 
# Activity year range
print(f"\n  Flight records span: "
      f"{flights[year_col].min()} - {flights[year_col].max()}")
flights_per_year = flights.groupby(year_col)[id_col].nunique()
print("\n  Unique active members per year:")
print(flights_per_year.to_string())
 


STEP 8 — ACTIVITY PATTERNS
  Members with zero total flights: 1,570
  [DECISION] 1570 members with zero total flights over full period → Kept; tagged as 'never_flew' flag in loyalty table
  'never_flew_flag' added to loyalty table.

  Flight records span: 2017 - 2018

  Unique active members per year:
year
2017    15766
2018    16737


In [23]:
loyalty.head()

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month,enrollment_date,cancellation_date,never_flew_flag
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN,2016-02-01,NaT,0
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,Unknown,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN,2016-03-01,NaT,0
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,Unknown,Single,Star,3839.75,Standard,2014,7,2018.0,1.0,2014-07-01,2018-01-01,0
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,Unknown,Single,Star,3839.75,Standard,2013,2,NaN,NaN,2013-02-01,NaT,0
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,NaN,NaN,2014-10-01,NaT,0


In [24]:
print("\n" + "="*70)
print("STEP 9 — SAVING EXPLORATORY PLOTS")
print("="*70)
 
# Plot 1: CLV distribution
if clv_col:
    fig, axes = plt.subplots(1, 2)
    loyalty[clv_col].plot(kind="hist", bins=50, ax=axes[0], color="steelblue",
                          edgecolor="white")
    axes[0].set_title("CLV Distribution (raw)")
    axes[0].set_xlabel("CLV")
 
    loyalty[clv_col].clip(lower=0).apply(np.log1p).plot(
        kind="hist", bins=50, ax=axes[1], color="coral", edgecolor="white"
    )
    axes[1].set_title("CLV Distribution (log scale)")
    axes[1].set_xlabel("log(CLV + 1)")
    plt.suptitle("Customer Lifetime Value", fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/plot_clv_distribution.png", dpi=120)
    plt.close()
    print(f"  Saved: {OUTPUT_DIR}/plot_clv_distribution.png")
 
# Plot 2: Active members over time
fig, ax = plt.subplots()
flights_per_year.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Active Members Per Year")
ax.set_xlabel("Year")
ax.set_ylabel("Unique Members")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_active_members_per_year.png", dpi=120)
plt.close()
print(f"  Saved: {OUTPUT_DIR}/plot_active_members_per_year.png")
 
# Plot 3: Card tier distribution (if exists)
tier_col = [c for c in loyalty.columns if "tier" in c or "card" in c or "loyalty_card" in c]
if tier_col:
    tier_col = tier_col[0]
    fig, ax = plt.subplots(figsize=(8, 4))
    loyalty[tier_col].value_counts().plot(kind="bar", ax=ax,
                                           color="teal", edgecolor="white")
    ax.set_title("Members by Card Tier")
    ax.set_xlabel("Tier")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/plot_card_tier_distribution.png", dpi=120)
    plt.close()
    print(f"  Saved: {OUTPUT_DIR}/plot_card_tier_distribution.png")
 
# Plot 4: Monthly flight volume over time
monthly_flights = (flights
                   .groupby("activity_date")[flights_col]
                   .sum()
                   .reset_index())
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(monthly_flights["activity_date"], monthly_flights[flights_col],
        color="steelblue", linewidth=1.2)
ax.set_title("Total Flights Booked Per Month (All Members)")
ax.set_xlabel("Month")
ax.set_ylabel("Total Flights")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_monthly_flight_volume.png", dpi=120)
plt.close()
print(f"  Saved: {OUTPUT_DIR}/plot_monthly_flight_volume.png")
 


STEP 9 — SAVING EXPLORATORY PLOTS
  Saved: Outputs/plot_clv_distribution.png
  Saved: Outputs/plot_active_members_per_year.png
  Saved: Outputs/plot_card_tier_distribution.png
  Saved: Outputs/plot_monthly_flight_volume.png


In [25]:
print("\n" + "="*70)
print("STEP 10 — SAVING CLEANED DATA")
print("="*70)
 
loyalty_out = f"{OUTPUT_DIR}/loyalty_cleaned.csv"
flights_out = f"{OUTPUT_DIR}/flights_cleaned.csv"
 
loyalty.to_csv(loyalty_out, index=False)
flights.to_csv(flights_out, index=False)
 
print(f"  Saved: {loyalty_out}  ({loyalty.shape[0]:,} rows × {loyalty.shape[1]} cols)")
print(f"  Saved: {flights_out}  ({flights.shape[0]:,} rows × {flights.shape[1]} cols)")


STEP 10 — SAVING CLEANED DATA
  Saved: Outputs/loyalty_cleaned.csv  (16,737 rows × 19 cols)
  Saved: Outputs/flights_cleaned.csv  (389,065 rows × 10 cols)


In [26]:
print("\n" + "="*70)
print("STEP 11 — CLEANING DECISIONS LOG")
print("="*70)
 
log_df = pd.DataFrame(cleaning_log)
log_path = f"{OUTPUT_DIR}/cleaning_decisions_log.csv"
log_df.to_csv(log_path, index=False)
print(f"\n  {len(cleaning_log)} decisions logged → {log_path}")
print("\n" + log_df.to_string(index=False))
 


STEP 11 — CLEANING DECISIONS LOG

  7 decisions logged → Outputs/cleaning_decisions_log.csv

                                                Issue                                                   Decision                                                                                                                Reason
                           Missing salary (4238 rows)                             Filled with 'Unknown' category                                               Salary band is categorical; imputing median would imply false precision
                               Null cancellation_year                     Kept as null — indicates active member                                             Null cancellation date means member has NOT cancelled; this is valid data
                     No single date column in flights      Combined year + month into activity_date (YYYY-MM-01)                                                          Required for time-series feature engineerin

In [27]:
print("\n" + "="*70)
print("PHASE 1 COMPLETE — SUMMARY")
print("="*70)
print(f"""
  Cleaned loyalty table  : {loyalty.shape}
  Cleaned flights table  : {flights.shape}
  Cleaning decisions     : {len(cleaning_log)}
  Plots saved            : {OUTPUT_DIR}/
 
  NEXT STEP → Phase 2: Churn Definition & Feature Engineering
  Run: python phase2_feature_engineering.py
""")


PHASE 1 COMPLETE — SUMMARY

  Cleaned loyalty table  : (16737, 19)
  Cleaned flights table  : (389065, 10)
  Cleaning decisions     : 7
  Plots saved            : Outputs/
 
  NEXT STEP → Phase 2: Churn Definition & Feature Engineering
  Run: python phase2_feature_engineering.py

